In [ ]:
%matplotlib inline

import sys
from functools import partial
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "common").is_dir())
RMSNORM_DIR = ROOT / "02-rmsnorm"
sys.path[:0] = [str(ROOT), str(RMSNORM_DIR)]

import matplotlib.pyplot as plt
import torch

from common.asm import dump_asm
from common.inputs import make_inputs
from common.run import make_runner, run_benchmark
from fvelikon import cute_rmsnorm as fvelikon_cute
from m4xig1 import cute_rmsnorm, triton_rmsnorm, triton_rmsnorm_optimized
from neuroslop.visualize_bench_results import plot_benchmark

M, N = 4096, 4096
EPS = cute_rmsnorm.EPS


def rmsnorm(x, w):
    return torch.nn.functional.rms_norm(x, (x.shape[-1],), w, EPS)

In [ ]:
AUTOTUNE = True  # False runs each kernel with its own defaults


def cute_vector_runner(width, autotune):
    selected = cute_rmsnorm.rmsnorm_vec(autotune=autotune)

    def specialize():
        call, asm = selected()
        return partial(call, WIDTH=width), asm

    return make_runner("cute", specialize)


def kernel_runners(autotune):
    triton_kernel = triton_rmsnorm.rmsnorm_kernel(autotune=autotune)
    triton_optimized_kernel = triton_rmsnorm_optimized.rmsnorm_kernel(autotune=autotune)
    runners = {
        "triton": make_runner(
            "triton", triton_kernel,
            extra_args=lambda x, w, out: (x.stride(0), x.shape[1], EPS),
            grid=lambda meta: (meta["x_ptr"].shape[0],),
        ),
        "triton_optimized": make_runner(
            "triton", triton_optimized_kernel,
            extra_args=lambda x, w, out: (x.stride(0), out.stride(0), *x.shape, EPS),
            grid=triton_rmsnorm_optimized.rmsnorm_grid,
        ),
        "cute": make_runner("cute", cute_rmsnorm.rmsnorm(autotune=autotune)),
        "cute_vec_w2": cute_vector_runner(2, autotune),
        "cute_vec_w4": cute_vector_runner(4, autotune),
        "fvelikon_cute_vec": make_runner("cute", fvelikon_cute.rmsnorm_vec(autotune=autotune)),
        "cuda": make_runner("cuda", RMSNORM_DIR / "fvelikon" / "cuda_rmsnorm.cu", entry="rmsnorm"),
    }
    return runners, (triton_kernel, triton_optimized_kernel)


VECTOR_WIDTHS = {"cute_vec_w2": 2, "cute_vec_w4": 4, "fvelikon_cute_vec": 4}
custom_kernels, TRITON_KERNELS = kernel_runners(AUTOTUNE)
IMPLEMENTATIONS = {
    "torch": make_runner("torch", rmsnorm),
    "torch_compile": make_runner(
        "torch_compile", lambda x, w: x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + EPS) * w
    ),
    **custom_kernels,
}

In [ ]:
# Check tails and vector alignment boundaries with the default launch configurations.
checks, _ = kernel_runners(autotune=False)
for m, n in [(1, 1), (3, 7), (5, 255), (2, 256), (7, 1000), (4, 4096), (3, 4098), (3, 8196)]:
    inputs = make_inputs((m, n), (n,))
    expected = rmsnorm(*inputs)
    for name, fn in checks.items():
        if n % VECTOR_WIDTHS.get(name, 1):
            continue
        torch.testing.assert_close(fn(*inputs), expected, msg=f"{name}, {m}x{n}")
print("OK")

In [ ]:
# Square, add, multiply by 1/rms, multiply by w.
inputs = make_inputs((M, N), (N,))
implementations = {
    name: fn for name, fn in IMPLEMENTATIONS.items() if N % VECTOR_WIDTHS.get(name, 1) == 0
}
results = run_benchmark(implementations, inputs, reference=rmsnorm, flops=4 * M * N)
plot_benchmark(results, baseline="torch_compile", title=f"{torch.cuda.get_device_name()} | {M}x{N}")
plt.show()
if AUTOTUNE:
    for name, kernel in zip(("triton", "triton_optimized"), TRITON_KERNELS):
        print(f"{name} config: {kernel.best_config}")
    for name in ("cute", *VECTOR_WIDTHS):
        if name in implementations:
            config = next(iter(implementations[name].asm().values()))["config"]
            print(f"{name} config: {config}")

In [ ]:
EXPORT_ASM = False
if EXPORT_ASM:
    for name, fn in implementations.items():
        listing = fn.asm()
        dump_asm(listing, RMSNORM_DIR / "m4xig1" / "asm" / "bench" / name)
        print(name, list(listing))